In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/prompt_predictor_v3"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5ForConditionalGeneration, T5Tokenizer
from sklearn.model_selection import train_test_split
from rouge_score import rouge_scorer
import numpy as np
import re

### Load Data

In this version, we filtered the data to eliminate things the model struggled with:
1. Code heavy answers
2. Long complex prompts or answers
3. Very short promprts or answers

In [ ]:
df = pd.read_csv("prompt_answer_pairs_clean.csv")
print(f"Original: {len(df)} rows")

# 1. Filter rows where answer is mostly code blocks to remove blocks
def clean_answer(text):
    text = re.sub(r'\[CODE_BLOCK_\d+\]', '', str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["answer"] = df["answer"].apply(clean_answer)
print(f"After code block filter: {len(df)} rows")

# 2. Drop rows where answer became too short after removing code blocks
df = df[df["answer"].str.split().str.len() >= 10]
print(f"After dropping short answers: {len(df)} rows")

# 3. Filter out very long prompts (over 50 words)
df = df[df["prompt"].str.split().str.len() <= 50]
print(f"After long prompt filter: {len(df)} rows")

# 4. Filter out very short prompts (less than 3 words)
df = df[df["prompt"].str.split().str.len() >= 3]
print(f"After short prompt filter: {len(df)} rows")

# 5. Filter out very long answers (over 400 words)
df = df[df["answer"].str.split().str.len() <= 400]
print(f"After long answer filter: {len(df)} rows")

df.to_csv("prompt_answer_pairs_filtered.csv", index=False, encoding="utf-8")

Original: 8058 rows
After code block filter: 8058 rows
After dropping short answers: 7509 rows
After long prompt filter: 4650 rows
After short prompt filter: 4621 rows
After long answer filter: 4315 rows


In this version we build context, building history from the prior turns

In [ ]:
def build_context_input(group):
    rows = group.reset_index(drop=True)
    context_inputs = []

    for i, row in rows.iterrows():
        # Only use last 2 prior turns
        history = ""
        start = max(0, i - 2)
        # Build history by appending previous
        for j in range(start, i):
            prev = rows.iloc[j]
            history += f"User: {prev['prompt']} Assistant: {prev['answer']} "

        # If there is a history prepend it, if not just use answer
        if history:
            context_input = f"predict prompt: {history.strip()} Current answer: {row['answer']}"
        else:
            context_input = f"predict prompt: {row['answer']}"

        context_inputs.append(context_input)

    group = group.copy()
    group["context_input"] = context_inputs
    return group

# Splits df into groups by conversation url and then applys the context function independently
df = df.sort_values(["chatgpt_url", "conv_index"]).reset_index(drop=True)
df = df.groupby("chatgpt_url", group_keys=False).apply(build_context_input)

df.to_csv("prompt_answer_pairs_context.csv", index=False, encoding="utf-8")

/tmp/ipykernel_1556/720515632.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("chatgpt_url", group_keys=False).apply(build_context_input)


In [ ]:
df = pd.read_csv("prompt_answer_pairs_context.csv")[["context_input", "prompt", "answer"]].dropna()

# Convert to strings
df["prompt"] = df["prompt"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()
df["context_input"] = df["context_input"].astype(str).str.strip()

# 70/15/15
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

### Tokenize data

In [ ]:
def tokenize_data(df):
  # Input is the answer, tokenizes answer text
  inputs = tokenizer(
      df["context_input"].tolist(),
      max_length=512,
      padding="max_length",
      truncation=True,
      return_tensors="pt"
    )
  # Target is the prompt, tokenizes prompt text
  targets = tokenizer(
      df["prompt"].tolist(),
      max_length=128,
      padding="max_length",
      truncation=True,
      return_tensors="pt"
  )

  labels = targets["input_ids"]
  # Replace padding tokens with -100 so model does not try to predict them
  labels[labels == tokenizer.pad_token_id] = -100

  # Create tokenized dataset
  dataset = torch.utils.data.TensorDataset(
      inputs["input_ids"],
      inputs["attention_mask"],
      labels
  )
  return dataset

In [ ]:
# Load tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")
# Model is a T5 language model
# T5 is a seq2seq model that is pretrained using a  using a denoising objective
model     = T5ForConditionalGeneration.from_pretrained("t5-small").to(torch.device("cuda"))

# Tokenize all datasets
train_dataset = tokenize_data(train_df)
val_dataset   = tokenize_data(val_df)
test_dataset  = tokenize_data(test_df)

# DataLoader does batching, shuffling and makes dataset iterable
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

### Evaluate how model is doing on validation set

In [ ]:
def evaluate(loader):
  # Switch model to evaluation mode
  model.eval()
  total_loss = 0
  # Do not track gradients to save memory
  with torch.no_grad():
    # For every batch accumulate the loss
    for batch in loader:
        input_ids, attention_mask, labels = batch
        input_ids      = input_ids.to(torch.device("cuda"))
        attention_mask = attention_mask.to(torch.device("cuda"))
        labels         = labels.to(torch.device("cuda"))
        loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
        total_loss += loss.item()
  # Return the average loss per epoch
  return total_loss / len(loader)

### Train Model

In this verision I added early stopping which stops training automatically when the validation loss stops improving to prevent overfitting. I did this instead of only doing 5 epochs like before to help model learn dataset better.

In [ ]:
# Load optimizer
optimizer   = torch.optim.AdamW(model.parameters(), lr=5e-5)
best_val    = float("inf")

# Early stopping settings
patience = 3 # how many epochs to wait before stopping
epochs_no_improve = 0
epoch = 0

while True:
  epoch += 1
  # Switch model to train mode
  model.train()
  total_train_loss = 0

  for i, batch in enumerate(train_loader):
    input_ids, attention_mask, labels = batch
    input_ids      = input_ids.to(torch.device("cuda"))
    attention_mask = attention_mask.to(torch.device("cuda"))
    labels         = labels.to(torch.device("cuda"))

    # Clear gradient from previous batch
    optimizer.zero_grad()
    # Calculates loss using T5 model
    loss = model(input_ids=input_ids,
                  attention_mask=attention_mask,
                  labels=labels).loss
    # Backpropagation
    loss.backward()
    # Gradient clipping (caps gradients at 1)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    # Updates model weights using gradients calculated by loss.backward
    optimizer.step()

    total_train_loss += loss.item()

    if (i + 1) % 50 == 0:
      print(f"  Epoch {epoch} | Step {i+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

  avg_train = total_train_loss / len(train_loader)
  avg_val   = evaluate(val_loader)
  print(f"\nEpoch {epoch} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")

  # Gets best model by updating model saved when average validation loss is less
  if avg_val < best_val:
    best_val = avg_val
    model.save_pretrained(SAVE_PATH)
    tokenizer.save_pretrained(SAVE_PATH)
    torch.save({
        "epoch": epoch,
        "best_val": best_val,
        "epochs_no_improve": epochs_no_improve,
        "optimizer_state": optimizer.state_dict()
    }, f"/content/drive/MyDrive/training_state.pt")
  # Break out of loop when no model improvement for 3 epochs in a row
  else:
    epochs_no_improve += 1
    if epochs_no_improve >= patience:
      print(f"\nEarly stopping triggered after {epoch} epochs")
      break

  Epoch 1 | Step 50/378 | Loss: 3.9948
  Epoch 1 | Step 100/378 | Loss: 3.8179
  Epoch 1 | Step 150/378 | Loss: 2.9108
  Epoch 1 | Step 200/378 | Loss: 3.2197
  Epoch 1 | Step 250/378 | Loss: 2.8987
  Epoch 1 | Step 300/378 | Loss: 3.5245
  Epoch 1 | Step 350/378 | Loss: 3.5583

Epoch 1 | Train Loss: 3.5097 | Val Loss: 3.2318


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 2 | Step 50/378 | Loss: 2.7439
  Epoch 2 | Step 100/378 | Loss: 3.6214
  Epoch 2 | Step 150/378 | Loss: 3.2490
  Epoch 2 | Step 200/378 | Loss: 3.1744
  Epoch 2 | Step 250/378 | Loss: 3.4588
  Epoch 2 | Step 300/378 | Loss: 3.4402
  Epoch 2 | Step 350/378 | Loss: 3.3629

Epoch 2 | Train Loss: 3.3045 | Val Loss: 3.1660


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 3 | Step 50/378 | Loss: 2.9446
  Epoch 3 | Step 100/378 | Loss: 2.8634
  Epoch 3 | Step 150/378 | Loss: 2.8402
  Epoch 3 | Step 200/378 | Loss: 2.9490
  Epoch 3 | Step 250/378 | Loss: 3.4034
  Epoch 3 | Step 300/378 | Loss: 3.0263
  Epoch 3 | Step 350/378 | Loss: 3.5420

Epoch 3 | Train Loss: 3.2080 | Val Loss: 3.1299


...

  Epoch 8 | Step 50/378 | Loss: 3.1574
  Epoch 8 | Step 100/378 | Loss: 2.9119
  Epoch 8 | Step 150/378 | Loss: 1.7950
  Epoch 8 | Step 200/378 | Loss: 2.8451
  Epoch 8 | Step 250/378 | Loss: 3.4107
  Epoch 8 | Step 300/378 | Loss: 3.1196
  Epoch 8 | Step 350/378 | Loss: 2.5701

Epoch 8 | Train Loss: 2.9011 | Val Loss: 3.0686


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 9 | Step 50/378 | Loss: 2.6254
  Epoch 9 | Step 100/378 | Loss: 2.6432
  Epoch 9 | Step 150/378 | Loss: 2.6894
  Epoch 9 | Step 200/378 | Loss: 2.1746
  Epoch 9 | Step 250/378 | Loss: 2.5030
  Epoch 9 | Step 300/378 | Loss: 3.3879
  Epoch 9 | Step 350/378 | Loss: 3.1778

Epoch 9 | Train Loss: 2.8581 | Val Loss: 3.0583


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 10 | Step 50/378 | Loss: 2.7893
  Epoch 10 | Step 100/378 | Loss: 2.7482
  Epoch 10 | Step 150/378 | Loss: 2.7265
  Epoch 10 | Step 200/378 | Loss: 2.7610
  Epoch 10 | Step 250/378 | Loss: 3.1252
  Epoch 10 | Step 300/378 | Loss: 2.9132
  Epoch 10 | Step 350/378 | Loss: 3.7759

Epoch 10 | Train Loss: 2.8158 | Val Loss: 3.0647
  Epoch 11 | Step 50/378 | Loss: 2.9765
  Epoch 11 | Step 100/378 | Loss: 2.8186
  Epoch 11 | Step 150/378 | Loss: 3.3563
  Epoch 11 | Step 200/378 | Loss: 2.2357
  Epoch 11 | Step 250/378 | Loss: 2.8323
  Epoch 11 | Step 300/378 | Loss: 3.2035
  Epoch 11 | Step 350/378 | Loss: 3.1014

Epoch 11 | Train Loss: 2.7794 | Val Loss: 3.0613
  Epoch 12 | Step 50/378 | Loss: 3.1206
  Epoch 12 | Step 100/378 | Loss: 1.6258
  Epoch 12 | Step 150/378 | Loss: 2.8271
  Epoch 12 | Step 200/378 | Loss: 2.6777
  Epoch 12 | Step 250/378 | Loss: 3.5545
  Epoch 12 | Step 300/378 | Loss: 2.9018
  Epoch 12 | Step 350/378 | Loss: 2.9240

Epoch 12 | Train Loss: 2.7362 | Val Loss:

### Evaulate Model Performace with ROUGE
ROUGE is a metric that measures how simular predicted prompt is to real prompt by comparing n-gram overlap

- ROUGE-1: Counts overlap of individual words
- ROUGE-2: Counts overlap of word pairs
- ROUGE-L: Finds longest common sequence of words in order

In [ ]:
model = T5ForConditionalGeneration.from_pretrained(SAVE_PATH).to(torch.device("cuda"))
model.eval()

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
r1, r2, rL = [], [], []

for _, row in test_df.iterrows():
    # Tokenize input
    input_text = row["context_input"]
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(torch.device("cuda"))

    # Generate prediction
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

    # Decode prediction back to text
    predicted = tokenizer.decode(output[0], skip_special_tokens=True)

    # Score predicted vs real prompt
    scores = scorer.score(row["prompt"], predicted)
    r1.append(scores["rouge1"].fmeasure)
    r2.append(scores["rouge2"].fmeasure)
    rL.append(scores["rougeL"].fmeasure)

# Print results
print(f"ROUGE-1:  {np.mean(r1):.4f}")
print(f"ROUGE-2:  {np.mean(r2):.4f}")
print(f"ROUGE-L:  {np.mean(rL):.4f}")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

ROUGE-1:  0.3094
ROUGE-2:  0.1512
ROUGE-L:  0.2710


### Sample Predictions

In [ ]:
results = []
for _, row in test_df.iterrows():
    input_text = row["context_input"]
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(torch.device("cuda"))

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

    predicted = tokenizer.decode(output[0], skip_special_tokens=True)
    rouge_l   = scorer.score(row["prompt"], predicted)["rougeL"].fmeasure

    results.append({
        "prompt": row["answer"],
        "reference":  row["prompt"],
        "prediction": predicted
    })

results_df = pd.DataFrame(results)

# Sort by rouge_l to see best and worst predictions
results_df = results_df.sort_values("rouge_l", ascending=False).reset_index(drop=True)

# Save results to predictions.csv which contains real prompt, predicted prompt, answer, ROUGE-L
results_df["rouge_l"] = [scorer.score(r["reference"], r["prediction"])["rougeL"].fmeasurefor r in results]
results_df.to_csv("predictions_v3.csv", index=False, encoding="utf-8")
print(f"Saved {len(results_df)} predictions to predictions.csv")
print(f"\nBest ROUGE-L:  {results_df['rouge_l'].max():.4f}")
print(f"Worst ROUGE-L: {results_df['rouge_l'].min():.4f}")
print(f"Mean ROUGE-L:  {results_df['rouge_l'].mean():.4f}")

Saved 648 predictions to predictions.csv

Best ROUGE-L:  1.0000
Worst ROUGE-L: 0.0000
Mean ROUGE-L:  0.2710


Comparing ROUGE-L of prompts which include coding and prompts which include no coding

In [ ]:
# Split test set into code and non-code prompts
code_pattern = r'def |class |import |\bif\b|\bfor\b|\bwhile\b|{|}|=>|!=|=='

code_df    = test_df[test_df["prompt"].str.contains(code_pattern, regex=True)]
no_code_df = test_df[~test_df["prompt"].str.contains(code_pattern, regex=True)]

print(f"Code prompts:    {len(code_df)} rows")
print(f"Non-code prompts: {len(no_code_df)} rows")

Code prompts:    188 rows
Non-code prompts: 460 rows


In [ ]:
def get_rouge_l(df):
    scores = []
    for _, row in df.iterrows():
        input_text = row["context_input"]
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(torch.device("cuda"))

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_length=128,
                num_beams=4,
                early_stopping=True
            )

        predicted = tokenizer.decode(output[0], skip_special_tokens=True)
        score = scorer.score(row["prompt"], predicted)
        scores.append(score["rougeL"].fmeasure)
    return scores

code_scores    = get_rouge_l(code_df)
no_code_scores = get_rouge_l(no_code_df)

print(f"Code prompts:     {np.mean(code_scores):.4f}")
print(f"Non-code prompts: {np.mean(no_code_scores):.4f}")
print(f"Overall:          {np.mean(code_scores + no_code_scores):.4f}")

Code prompts:     0.2373
Non-code prompts: 0.2848
Overall:          0.2710
